# Silver Layer Cleaning and Validation Notebook

This notebook transforms raw Bronze-layer data into cleaned Silver-layer Delta tables for the parcel sorting hub data pipeline.

The Silver layer is responsible for:
- validating mandatory business keys and foreign keys,
- rejecting invalid records into quarantine tables,
- standardizing data types and string formats,
- deduplicating valid records,
- incrementally merging clean records into Silver tables.

The notebook follows the Medallion Architecture pattern:

Bronze -> Raw ingested data  
Silver -> Cleaned, validated, standardized data  
Quarantine -> Invalid records rejected from Silver processing

Only records that pass validation are eligible to be merged into the Silver layer.

# ORDER OF EXECUTION (table dependencies)

## Processing Order

Tables are processed in dependency order to support foreign key validation.

Parent tables are validated and merged into Silver before child tables that reference them.

For example:
1. `silver.intake` is processed before `silver.parcels`
2. `silver.loading` is processed before `silver.parcels`
3. `silver.parcels` is processed before child event tables such as scans, exceptions, sorting events, or status history

This order prevents valid child records from being incorrectly quarantined because their parent records have not yet been loaded into Silver.

## Tables Processing Order
### Dependency Step 1 (tables without FK's)
-     1. device
-     2. intake
-     3. loading

### Dependency Step 2 (tables with FK's - depends on step 1)
-     4. parcels        (FK -> intakes, loading)

### Dependency Step 3 (tables with FK's - depends on step 2)
-     5. scans          (FK -> parcels, devices)
-     6. exceptions     (FK -> parcels, devices)
-     7. sorting        (FK -> parcels, devices)
-     8. status_history (FK -> parcels)

## Incremental Processing

Each validation view reads only newly ingested Bronze records.

The incremental filter compares the Bronze `_ingested_at` timestamp against the maximum `_updated_at` value already present in the corresponding Silver table.

This prevents the pipeline from reprocessing records that were already merged into Silver during previous runs.

Example pattern:

```
WHERE _ingested_at > (
    SELECT COALESCE(MAX(_updated_at), '1970-01-01') FROM silver.<table_name>
)
```

# DATA VALIDATION

Validation occurs in 3 steps:

## Step 1 (Validation)
- Data and business logic validation with the use of `TEMP VIEW`
- Flag bad data with a description in `_rejection_reason`

## Step 2 (Quarantine Isolation)
- Flagged `bad data` (so the data where `_rejection_reason` is not `NULL`) is put in the `quarantine` table

## Step 3 (Deduplication and Merge clean data into Silver Table)
- Flagged `good data` (so the data where `_rejection_reason` is `NULL`) is put in the `silver` table

## DEPENDENCY STEP 1

### DEVICES

In [0]:
CREATE OR REPLACE TEMP VIEW validated_devices AS

SELECT  
    device_id,        
    type,            
    zone,             
    status, 

    CASE
    -- Primary Key Validation
        WHEN device_id IS NULL 
            OR TRIM(device_id) = '' 
        THEN 'NULL_OR_EMPTY_PK'

        WHEN UPPER(TRIM(device_id)) NOT LIKE 'DEV-%'                
        THEN 'INVALID_PK' 

        WHEN LOWER(TRIM(device_id)) = 'null'           
        THEN 'STRING_NULL_IN_PK'

    -- Status Validation
        WHEN status IS NULL 
            OR TRIM(status) = ''       
        THEN 'NULL_OR_EMPTY_STATUS'

        WHEN LOWER(TRIM(status)) = 'null'           
        THEN 'STRING_NULL_IN_STATUS'

        WHEN UPPER(TRIM(status)) NOT IN (
            'ACTIVE', 
            'MAINTENANCE'
        )                          
        THEN 'INVALID_STATUS'

    -- Type Validation  
        WHEN type IS NULL 
            OR TRIM(type) = ''           
        THEN 'NULL_OR_EMPTY_TYPE'

        WHEN LOWER(TRIM(type)) = 'null'
        THEN 'STRING_NULL_IN_TYPE'

        WHEN TRIM(type) NOT IN (
            'Sorter', 
            'Telescopic Conveyor', 
            'Scanner', 'Dimensioner'
        )                          
        THEN 'INVALID_TYPE'
        
    -- Zone Validation
        WHEN zone IS NULL 
            OR TRIM(zone) = ''           
        THEN 'NULL_OR_EMPTY_ZONE'

        WHEN LOWER(TRIM(zone)) = 'null'
        THEN 'STRING_NULL_IN_ZONE'

        WHEN TRIM(zone) NOT IN (
            'Sort_Area', 
            'Inbound', 
            'Outbound'
        )                          
        THEN 'INVALID_ZONE'

        ELSE NULL
    END AS _rejection_reason,

    _ingested_at,
    _source_file,
    current_timestamp() AS _validated_at

FROM parcel_sorting_hub.bronze.devices
WHERE _ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.devices);

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.devices
SELECT 
    device_id,        
    type,            
    zone,             
    status,
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_devices
WHERE _rejection_reason IS NOT NULL

In [0]:
MERGE INTO parcel_sorting_hub.silver.devices AS target
USING (
    WITH deduped AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY _ingested_at DESC) AS row_num_id,
            ROW_NUMBER() OVER (PARTITION BY 
                device_id,
                type,
                zone,
                status
            ORDER BY _ingested_at DESC) row_dup
        FROM validated_devices
        WHERE _rejection_reason IS NULL
        )
    
    SELECT
        TRIM(device_id)     AS device_id,
        TRIM(type)          AS type,
        TRIM(zone)          AS zone,
        TRIM(status)        AS status,
        current_timestamp() AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1 
    AND 
        row_dup = 1

) AS source
ON target.device_id = source.device_id
WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *

### INTAKE

In [0]:
CREATE OR REPLACE TEMP VIEW validated_intake AS

SELECT  
    delivery_id,        
    dock_id,            
    arrival_time,          
    unload_start,                    
    unload_end,
    parcel_count,
    truck_plate,
    source_hub_id,

    CASE
    -- Primary Key Validation
        WHEN delivery_id IS NULL
            OR TRIM(delivery_id) = ''
        THEN 'NULL_OR_EMPTY_PK'

        WHEN LOWER(TRIM(delivery_id)) = 'null'
        THEN 'STRING_NULL_IN_PK'

        WHEN UPPER(TRIM(delivery_id)) NOT LIKE 'DLV-%'
        THEN 'INVALID_PK'

    -- Foreign Key Validation
        WHEN dock_id IS NULL
            OR TRIM(dock_id) = ''
        THEN 'NULL_OR_EMPTY_DOCK_ID'

        WHEN LOWER(TRIM(dock_id)) = 'null'
        THEN 'STRING_NULL_IN_DOCK_ID'

        WHEN UPPER(TRIM(dock_id)) NOT LIKE 'IN-DOCK-%'
        THEN 'INVALID_FK'

    -- Timestamp Validation
        -- Arrival Time
        WHEN arrival_time IS NULL
            OR TRIM(arrival_time) = ''
        THEN 'NULL_OR_EMPTY_ARRIVAL_TIME'

        WHEN LOWER(TRIM(arrival_time)) = 'null'
        THEN 'STRING_NULL_IN_ARRIVAL_TIME'

        WHEN TRY_CAST(TRIM(arrival_time) AS TIMESTAMP) IS NULL
        THEN 'INVALID_TIMESTAMP_UNLOAD_START_TIME'

        -- Unload Start
        WHEN unload_start IS NULL
            OR TRIM(unload_start) = ''
        THEN 'NULL_OR_EMPTY_UNLOAD_START'

        WHEN LOWER(TRIM(unload_start)) = 'null'
        THEN 'STRING_NULL_IN_UNLOAD_START'

        WHEN TRY_CAST(TRIM(unload_start) AS TIMESTAMP) IS NULL
        THEN 'INVALID_TIMESTAMP_UNLOAD_START_TIME'

        -- Unload End
        WHEN unload_end IS NULL
            OR TRIM(unload_end) = ''
        THEN 'NULL_OR_EMPTY_UNLOAD_END'

        WHEN LOWER(TRIM(unload_end)) = 'null'
        THEN 'STRING_NULL_IN_UNLOAD_END'

        WHEN TRY_CAST(TRIM(unload_end) AS TIMESTAMP) IS NULL
        THEN 'INVALID_TIMESTAMP_UNLOAD_END_TIME'

    -- Business Logic Validation
        WHEN arrival_time > unload_start
        THEN 'INVALID_ARRIVAL_TIME'

        WHEN unload_start > unload_end
        THEN 'INVALID_UNLOAD_START'

        WHEN parcel_count < 0
        THEN 'INVALID_PARCEL_COUNT'

    -- Truck Plate Validation
        WHEN truck_plate IS NULL
            OR TRIM(truck_plate) = ''
        THEN 'NULL_OR_EMPTY_TRUCK_PLATE'

        WHEN LOWER(TRIM(truck_plate) ) = 'null'
        THEN 'STRING_NULL_IN_TRUCK_PLATE'

    -- Source Hub Validation
        WHEN source_hub_id IS NULL
            OR TRIM(source_hub_id) = ''
        THEN 'NULL_OR_EMPTY_SOURCE_HUB' 

        WHEN LOWER(TRIM(source_hub_id)) = 'null'
        THEN 'STRING_NULL_IN_SOURCE_HUB'

        ELSE NULL
    END AS _rejection_reason,

    _ingested_at,
    _source_file,
    current_timestamp() AS _validated_at

FROM parcel_sorting_hub.bronze.intake
WHERE _ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.intake);

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.intake
SELECT 
    delivery_id,        
    dock_id,            
    arrival_time,          
    unload_start,                    
    unload_end,
    parcel_count,
    truck_plate,
    source_hub_id,
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_intake
WHERE _rejection_reason IS NOT NULL;

In [0]:
MERGE INTO parcel_sorting_hub.silver.intake AS target
USING (
    WITH deduped AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY delivery_id ORDER BY _ingested_at DESC) AS row_num_id,
            ROW_NUMBER() OVER (PARTITION BY 
                delivery_id,
                dock_id,
                arrival_time,
                unload_start,
                unload_end,
                parcel_count,
                truck_plate,
                source_hub_id
            ORDER BY _ingested_at DESC) row_dup
        FROM validated_intake
        WHERE _rejection_reason IS NULL
        )
    
    
    SELECT
        TRIM(delivery_id)                           AS delivery_id,
        TRIM(dock_id)                               AS dock_id,
        TRY_CAST(TRIM(arrival_time) AS TIMESTAMP)   AS arrival_time,
        TRY_CAST(TRIM(unload_start) AS TIMESTAMP)   AS unload_start,
        TRY_CAST(TRIM(unload_end) AS TIMESTAMP)     AS unload_end,
        TRIM(parcel_count)                          AS parcel_count,
        TRIM(truck_plate)                           AS truck_plate,
        TRIM(source_hub_id)                         AS source_hub_id,
        current_timestamp()                         AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1 
    AND 
        row_dup = 1

) AS source
ON target.delivery_id = source.delivery_id
WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *

### LOADING

In [0]:
CREATE OR REPLACE TEMP VIEW validated_loading AS

SELECT  
    loading_id,        
    dock_id,            
    start_time,             
    close_time,                    
    parcel_count,
    route_id,
    employee_id,

    CASE
    -- Primary Key Validation
        WHEN loading_id IS NULL 
            OR TRIM(loading_id) = '' 
        THEN 'NULL_OR_EMPTY_PK'

        WHEN LOWER(TRIM(loading_id)) = 'null' 
        THEN 'STRING_NULL_IN_PK'

        WHEN UPPER(TRIM(loading_id)) NOT LIKE 'LOD-%'
        THEN 'INVALID_PK'

    -- Foreign Key Validation
        WHEN dock_id IS NULL 
            OR TRIM(dock_id) = '' 
        THEN 'NULL_OR_EMPTY_DOCK_ID'

        WHEN LOWER(TRIM(dock_id)) = 'null' 
        THEN 'STRING_NULL_IN_DOCK_ID'

        WHEN UPPER(TRIM(dock_id)) NOT LIKE 'OUT-DOCK-%'
        THEN 'INVALID_FK'

    -- Timestamp Validation
        WHEN start_time IS NULL 
            OR TRIM(start_time) = '' 
        THEN 'NULL_OR_EMPTY_START_TIME'

        WHEN LOWER(TRIM(start_time)) = 'null' 
        THEN 'STRING_NULL_IN_START_TIME'

        WHEN close_time IS NULL 
            OR TRIM(close_time) = '' 
        THEN 'NULL_OR_EMPTY_CLOSE_TIME'

        WHEN LOWER(TRIM(close_time)) = 'null' 
        THEN 'STRING_NULL_IN_CLOSE_TIME'

    -- Parcel Validation
        WHEN parcel_count IS NULL 
            OR TRIM(parcel_count) = '' 
        THEN 'NULL_OR_EMPTY_PARCEL_COUNT'

        WHEN LOWER(TRIM(parcel_count)) = 'null' 
        THEN 'STRING_NULL_IN_PARCEL_COUNT'

    -- Business Logic Validation
        WHEN start_time > close_time
        THEN 'INVALID_START_TIME'

        WHEN parcel_count < 0
        THEN 'INVALID_PARCEL_COUNT'

    -- Route ID Validation
        WHEN route_id IS NULL 
            OR TRIM(route_id) = '' 
        THEN 'NULL_OR_EMPTY_ROUTE_ID'

        WHEN LOWER(TRIM(route_id)) = 'null' 
        THEN 'STRING_NULL_IN_ROUTE_ID'

    -- Employee ID Validation
        WHEN employee_id IS NULL 
            OR TRIM(employee_id) = '' 
        THEN 'NULL_OR_EMPTY_EMPLOYEE_ID'

        WHEN LOWER(TRIM(employee_id)) = 'null' 
        THEN 'STRING_NULL_IN_EMPLOYEE_ID'

        WHEN UPPER(TRIM(employee_id)) NOT LIKE 'EMP-%'
        THEN 'INVALID_EMPLOYEE_ID'

        ELSE NULL
    END AS _rejection_reason,

    _ingested_at,
    _source_file,
    current_timestamp() AS _validated_at

FROM parcel_sorting_hub.bronze.loading
WHERE _ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.loading);

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.loading
SELECT 
    loading_id,        
    dock_id,            
    start_time,             
    close_time,                    
    parcel_count,
    route_id,
    employee_id,
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_loading
WHERE _rejection_reason IS NOT NULL;

In [0]:
MERGE INTO parcel_sorting_hub.silver.loading AS target
USING (
    WITH deduped AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY loading_id ORDER BY _ingested_at DESC) AS row_num_id,
            ROW_NUMBER() OVER (PARTITION BY 
                loading_id,
                dock_id,
                start_time,
                close_time,
                parcel_count,
                route_id,
                employee_id
            ORDER BY _ingested_at DESC) row_dup
        FROM validated_loading
        WHERE _rejection_reason IS NULL
    )
    
    SELECT
        TRIM(loading_id)                        AS loading_id,
        TRIM(dock_id)                           AS dock_id,
        TRY_CAST(TRIM(start_time) AS TIMESTAMP) AS start_time,
        TRY_CAST(TRIM(close_time) AS TIMESTAMP) AS close_time,
        TRIM(parcel_count)                      AS parcel_count,
        TRIM(route_id)                          AS route_id,
        TRIM(employee_id)                       AS employee_id,
        current_timestamp()                     AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1 
    AND 
        row_dup = 1

) AS source
ON target.loading_id = source.loading_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

## DEPENDENCY STEP 2

### PARCELS

In [0]:
CREATE OR REPLACE TEMP VIEW validated_parcels AS

SELECT  
    p.parcel_id,        
    p.status,            
    p.weight_kg,             
    p.length_cm,                    
    p.width_cm,
    p.height_cm,
    p.service_type,
    p.received_at,
    p.source_hub_id,
    p.delivery_id,
    p.destination_hub_id,
    p.loading_id,

    CASE
    -- Primary Key Validation
        WHEN p.parcel_id IS NULL 
          OR TRIM(p.parcel_id) = ''          
        THEN 'NULL_OR_EMPTY_PK'

        WHEN LOWER(TRIM(p.parcel_id)) = 'null'                      
        THEN 'STRING_NULL_IN_PK'

        WHEN LENGTH(TRIM(p.parcel_id)) != 24
        THEN 'INVALID_PK'

    -- Foreign Key Validation
        -- Source Hub ID Validation   
        WHEN p.source_hub_id IS NULL 
          OR TRIM(p.source_hub_id) = ''  
        THEN 'NULL_OR_EMPTY_SOURCE_HUB_ID'

        WHEN LOWER(TRIM(p.source_hub_id)) = 'null'                  
        THEN 'STRING_NULL_IN_SOURCE_HUB_ID'

        -- Delivery ID Validation
        WHEN p.delivery_id IS NULL 
          OR TRIM(p.delivery_id) = ''      
        THEN 'NULL_OR_EMPTY_DELIVERY_ID'

        WHEN LOWER(TRIM(p.delivery_id)) = 'null'                    
        THEN 'STRING_NULL_IN_DELIVERY_ID'

        WHEN UPPER(TRIM(p.delivery_id)) NOT LIKE 'DLV-%'
        THEN 'INVALID_FK_DELIVERY'

        -- Destination Hub ID Validation
        WHEN p.destination_hub_id IS NULL 
          OR TRIM(p.destination_hub_id) = ''                      
        THEN 'NULL_OR_EMPTY_DESTINATION_HUB_ID'

        WHEN LOWER(TRIM(p.destination_hub_id)) = 'null'             
        THEN 'STRING_NULL_IN_DESTINATION_HUB_ID'

        -- Loading ID Validation
        WHEN p.loading_id IS NULL 
          OR TRIM(p.loading_id) = ''        
        THEN 'NULL_OR_EMPTY_LOADING_ID'

        WHEN LOWER(TRIM(p.loading_id)) = 'null'                     
        THEN 'STRING_NULL_IN_LOADING_ID'

        WHEN UPPER(TRIM(p.loading_id)) NOT LIKE 'LOD-%'
        THEN 'INVALID_FK_LOADING'

    -- Numeric Validation
        WHEN p.weight_kg IS NULL 
          OR TRIM(p.weight_kg) = '' 
        THEN 'NULL_OR_EMPTY_WEIGHT_KG'

        WHEN LOWER(TRIM(p.weight_kg)) = 'null' 
        THEN 'STRING_NULL_IN_WEIGHT_KG'

        WHEN TRY_CAST(TRIM(p.weight_kg) AS DECIMAL(10,2)) IS NULL
        THEN 'INVALID_WEIGHT_KG_FORMAT'

        WHEN TRY_CAST(TRIM(p.weight_kg) AS DECIMAL(10,2)) <= 0
        THEN 'INVALID_WEIGHT_KG'

        WHEN p.length_cm IS NULL 
          OR TRIM(p.length_cm) = '' 
        THEN 'NULL_OR_EMPTY_LENGTH_CM'

        WHEN LOWER(TRIM(p.length_cm)) = 'null' 
        THEN 'STRING_NULL_IN_LENGTH_CM'

        WHEN TRY_CAST(TRIM(p.length_cm) AS DECIMAL(10,2)) IS NULL
        THEN 'INVALID_LENGTH_CM_FORMAT'

        WHEN TRY_CAST(TRIM(p.length_cm) AS DECIMAL(10,2)) <= 0
        THEN 'INVALID_LENGTH_CM'

        WHEN p.width_cm IS NULL 
          OR TRIM(p.width_cm) = '' 
        THEN 'NULL_OR_EMPTY_WIDTH_CM'

        WHEN LOWER(TRIM(p.width_cm)) = 'null' 
        THEN 'STRING_NULL_IN_WIDTH_CM'

        WHEN TRY_CAST(TRIM(p.width_cm) AS DECIMAL(10,2)) IS NULL
        THEN 'INVALID_WIDTH_CM_FORMAT'

        WHEN TRY_CAST(TRIM(p.width_cm) AS DECIMAL(10,2)) <= 0
        THEN 'INVALID_WIDTH_CM'

        WHEN p.height_cm IS NULL 
          OR TRIM(p.height_cm) = '' 
        THEN 'NULL_OR_EMPTY_HEIGHT_CM'

        WHEN LOWER(TRIM(p.height_cm)) = 'null' 
        THEN 'STRING_NULL_IN_HEIGHT_CM'

        WHEN TRY_CAST(TRIM(p.height_cm) AS DECIMAL(10,2)) IS NULL
        THEN 'INVALID_HEIGHT_CM_FORMAT'

        WHEN TRY_CAST(TRIM(p.height_cm) AS DECIMAL(10,2)) <= 0
        THEN 'INVALID_HEIGHT_CM'

    -- Service Type Validation
        WHEN p.service_type IS NULL 
          OR TRIM(p.service_type) = '' 
        THEN 'NULL_OR_EMPTY_SERVICE_TYPE'

        WHEN LOWER(TRIM(p.service_type)) = 'null' 
        THEN 'STRING_NULL_IN_SERVICE_TYPE'

        WHEN UPPER(TRIM(p.service_type)) NOT IN (
            'PRIORITY', 
            'LOCKER_24H', 
            'COURIER_STD'
        )
        THEN 'INVALID_SERVICE_TYPE'

    -- Received At Validation
        WHEN p.received_at IS NULL 
          OR TRIM(p.received_at) = '' 
        THEN 'NULL_OR_EMPTY_RECEIVED_AT'

        WHEN LOWER(TRIM(p.received_at)) = 'null' 
        THEN 'STRING_NULL_IN_RECEIVED_AT'

        WHEN TRY_CAST(TRIM(p.received_at) AS TIMESTAMP) IS NULL
        THEN 'INVALID_RECEIVED_AT'

    -- Status Validation
        WHEN p.status IS NULL 
          OR TRIM(p.status) = '' 
        THEN 'NULL_OR_EMPTY_STATUS'

        WHEN LOWER(TRIM(p.status)) = 'null' 
        THEN 'STRING_NULL_IN_STATUS'

        WHEN UPPER(TRIM(p.status)) NOT IN (
            'LOADED', 
            'EXCEPTION'
        )
        THEN 'INVALID_STATUS'

    -- Foreign Key Presence Validation
        WHEN i.delivery_id IS NULL                                  
        THEN 'FK_DELIVERY_ID_NOT_FOUND_IN_SILVER'

        WHEN l.loading_id IS NULL                                   
        THEN 'FK_LOADING_ID_NOT_FOUND_IN_SILVER'

        ELSE NULL
    END AS _rejection_reason,

    p._ingested_at,
    p._source_file,
    current_timestamp() AS _validated_at

FROM parcel_sorting_hub.bronze.parcels p

LEFT JOIN parcel_sorting_hub.silver.intake i 
    ON TRIM(p.delivery_id) = TRIM(i.delivery_id)

LEFT JOIN parcel_sorting_hub.silver.loading l  
    ON TRIM(p.loading_id) = TRIM(l.loading_id)

WHERE p._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.parcels) 


In [0]:
INSERT INTO parcel_sorting_hub.quarantine.parcels
SELECT 
    parcel_id,        
    status,            
    weight_kg,             
    length_cm,                    
    width_cm,
    height_cm,
    service_type,
    received_at,
    source_hub_id,
    delivery_id,
    destination_hub_id,
    loading_id,
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_parcels
WHERE _rejection_reason IS NOT NULL;

In [0]:
MERGE INTO parcel_sorting_hub.silver.parcels AS target
USING (
    WITH deduped AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY TRIM(parcel_id) ORDER BY _ingested_at DESC) AS row_num_id,
            ROW_NUMBER() OVER (PARTITION BY 
                TRIM(parcel_id),
                TRIM(status),
                TRIM(weight_kg),
                TRIM(length_cm),
                TRIM(width_cm),
                TRIM(height_cm),
                TRIM(service_type),
                TRIM(received_at),
                TRIM(source_hub_id),
                TRIM(destination_hub_id),
                TRIM(loading_id)
             ORDER BY _ingested_at DESC) row_dup
        FROM validated_parcels
        WHERE _rejection_reason IS NULL
    )
    
    SELECT
        TRIM(parcel_id)                             AS parcel_id,
        TRIM(status)                                AS status,
        TRY_CAST(TRIM(weight_kg) AS DECIMAL(10,2))  AS weight_kg,
        TRY_CAST(TRIM(length_cm) AS DECIMAL(10,2))  AS length_cm,
        TRY_CAST(TRIM(width_cm) AS DECIMAL(10,2))   AS width_cm,
        TRY_CAST(TRIM(height_cm) AS DECIMAL(10,2))  AS height_cm,
        TRIM(service_type)                          AS service_type,
        TRY_CAST(TRIM(received_at) AS TIMESTAMP)    AS received_at,
        TRIM(source_hub_id)                         AS source_hub_id,
        TRIM(destination_hub_id)                    AS destination_hub_id,
        TRIM(loading_id)                            AS loading_id,
        current_timestamp()                         AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1 
    AND 
        row_dup = 1

) AS source
ON target.parcel_id = source.parcel_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

## DEPENDENCY STEP 3

### SCANS

In [0]:
CREATE OR REPLACE TEMP VIEW validated_scans AS 

SELECT
    s.scan_id,        
    s.parcel_id,            
    s.device_id,             
    s.scan_type,   
    s.result,
    s.scan_timestamp,
    s.dynamic_weight,

    CASE
    -- Primary Key Validation
        WHEN s.scan_id IS NULL 
            OR TRIM(s.scan_id) = ''      
        THEN 'NULL_OR_EMPTY_PK'

        WHEN LOWER(TRIM(s.scan_id)) = 'null'                
        THEN 'STRING_NULL_IN_PK'

        WHEN s.scan_id NOT LIKE 'SCN-%'
        THEN 'INVALID_PK_FORMAT'

    -- Foreign Key Validation
        -- Parcel ID
        WHEN s.parcel_id IS NULL 
          OR TRIM(s.parcel_id) = ''          
        THEN 'NULL_OR_EMPTY_FK_PARCEL_ID'

        WHEN LOWER(TRIM(p.parcel_id)) = 'null'                      
        THEN 'STRING_NULL_IN_FK_PARCEL_ID'

        WHEN LENGTH(TRIM(p.parcel_id)) != 24
        THEN 'INVALID_FK_PARCEL_ID'

        -- Device ID
        WHEN s.device_id IS NULL 
            OR TRIM(s.device_id) = ''  
        THEN 'NULL_OR_EMPTY_DEVICE_ID'

        WHEN LOWER(TRIM(s.device_id)) = 'null'              
        THEN 'STRING_NULL_IN_DEVICE_ID'

        WHEN UPPER(TRIM(s.device_id)) NOT LIKE 'DEV-%'                
        THEN 'INVALID_FK_DEVICE_ID' 

    -- Scan Type Validation
        WHEN TRIM(s.scan_type) IS NULL 
          OR TRIM(s.scan_type) = '' 
        THEN 'NULL_OR_EMPTY_SCAN_TYPE'

        WHEN LOWER(TRIM(s.scan_type)) = 'null'
        THEN 'STRING_NULL_IN_SCAN_TYPE'

        WHEN UPPER(TRIM(s.scan_type)) NOT IN (
            'LOADED', 
            'INBOUND', 
            'LOADING_DOCK', 
            'DIMENSIONING', 
            'DESTINATION'
            )
        THEN 'INVALID_SCAN_TYPE'

    -- Result Validation  
        WHEN TRIM(s.result) IS NULL 
          OR TRIM(s.result) = '' 
        THEN 'NULL_OR_EMPTY_RESULT'

        WHEN LOWER(TRIM(s.result)) = 'null'
        THEN 'STRING_NULL_IN_RESULT'

        WHEN s.result NOT IN (
            'OK', 
            'FAIL'
            )
        THEN 'INVALID_RESULT'

    -- Timestamp Validation
        WHEN TRIM(s.scan_timestamp) IS NULL 
          OR TRIM(s.scan_timestamp) = '' 
        THEN 'NULL_OR_EMPTY_SCAN_TIMESTAMP'

        WHEN TRY_CAST(TRIM(s.scan_timestamp) AS TIMESTAMP) IS NULL
        THEN 'INVALID_TIMESTAMP_FORMAT'

    -- Numeric Validation
        -- Dynamic Weight
        WHEN s.dynamic_weight IS NULL 
          OR TRIM(s.dynamic_weight) = '' 
        THEN 'NULL_OR_EMPTY_DYNAMIC_WEIGHT'

        WHEN LOWER(TRIM(s.dynamic_weight)) = 'null'
        THEN 'STRING_NULL_IN_DYNAMIC_WEIGHT'

        WHEN TRY_CAST(TRIM(s.dynamic_weight) AS DECIMAL(10,2)) IS NULL
        THEN 'INVALID_DYNAMIC_WEIGHT_FORMAT'

        WHEN TRY_CAST(TRIM(s.dynamic_weight) AS DECIMAL(10,2)) <= 0
        THEN 'INVALID_DYNAMIC_WEIGHT'      

    -- Foreign Key Presence Validation
        WHEN d.device_id IS NULL                            
        THEN 'FK_DEVICE_NOT_FOUND_IN_SILVER'

        WHEN p.parcel_id IS NULL                            
        THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'

        ELSE NULL
    END AS _rejection_reason,
    
    _ingested_at,
    _source_file,
    current_timestamp() AS _validated_at
    
FROM parcel_sorting_hub.bronze.scans s

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON TRIM(s.parcel_id) = TRIM(p.parcel_id)

LEFT JOIN parcel_sorting_hub.silver.devices d  
    ON TRIM(s.device_id) = TRIM(d.device_id)

WHERE s._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.scans)

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.scans
SELECT 
    scan_id,        
    parcel_id,            
    device_id,             
    scan_type,   
    result,
    scan_timestamp,
    dynamic_weight,
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_scans 
WHERE _rejection_reason IS NOT NULL;

In [0]:
MERGE INTO parcel_sorting_hub.silver.scans AS target
USING (
    WITH deduped AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY TRIM(scan_id) ORDER BY _ingested_at DESC) AS row_num_id,
            ROW_NUMBER() OVER (PARTITION BY 
                TRIM(scan_id),
                TRIM(parcel_id),
                TRIM(device_id),
                TRIM(scan_type),
                TRIM(result),
                TRIM(scan_timestamp),
                TRIM(dynamic_weight)
            ORDER BY _ingested_at DESC) row_dup
        FROM validated_scans
        WHERE _rejection_reason IS NULL
    )
    
    SELECT
        TRIM(scan_id)                                   AS scan_id,
        TRIM(parcel_id)                                 AS parcel_id,
        TRIM(device_id)                                 AS device_id,
        TRIM(scan_type)                                 AS scan_type,
        TRIM(result)                                    AS result,
        TRY_CAST(TRIM(scan_timestamp) AS TIMESTAMP)     AS scan_timestamp,
        TRY_CAST(TRIM(dynamic_weight) AS DECIMAL(10,2)) AS dynamic_weight,
        current_timestamp()                             AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1 
    AND 
        row_dup = 1    

) AS source
ON target.scan_id = source.scan_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

### EXCEPTIONS

In [0]:
CREATE OR REPLACE TEMP VIEW validated_exceptions AS 

SELECT 
    e.exception_id,        
    e.parcel_id,            
    e.device_id,             
    e.error_code,   
    e.status,
    e.reported_at,
    e.resolved_at,
    e.employee_id,  

    CASE
    -- Primary Key Validation
        -- Exception ID
        WHEN e.exception_id IS NULL 
            OR TRIM(e.exception_id) = '' 
        THEN 'NULL_OR_EMPTY_PK'

        WHEN LOWER(TRIM(e.exception_id)) = 'null'                
        THEN 'STRING_NULL_IN_PK'

    -- Foreign Key Validation
        -- Parcel ID
        WHEN e.parcel_id IS NULL 
            OR TRIM(e.parcel_id) = '' 
        THEN 'NULL_OR_EMPTY_FK_PARCEL_ID'

        WHEN LOWER(TRIM(e.parcel_id)) = 'null'                
        THEN 'STRING_NULL_IN_FK_PARCEL_ID'

        WHEN LENGTH(TRIM(e.parcel_id)) != 24
        THEN 'INVALID_FK_PARCEL_ID'

        -- Device ID
        WHEN e.device_id IS NULL 
            OR TRIM(e.device_id) = ''       
        THEN 'NULL_OR_EMPTY_FK_DEVICE_ID'

        WHEN LOWER(TRIM(e.device_id)) = 'null'                   
        THEN 'STRING_NULL_IN_FK_DEVICE_ID' 

        WHEN UPPER(TRIM(e.device_id)) NOT LIKE 'DEV-%'                
        THEN 'INVALID_FK_DEVICE_ID'

    -- Error Code Validation
        WHEN e.error_code IS NULL 
            OR TRIM(e.error_code) = ''       
        THEN 'NULL_OR_EMPTY_ERROR_CODE'

        WHEN LOWER(TRIM(e.error_code)) = 'null'                   
        THEN 'STRING_NULL_IN_ERROR_CODE'

        WHEN UPPER(TRIM(e.error_code)) NOT IN (
            'WRONG_HUB_ROUTING',
            'LOW_CONTRAST_LABEL',
            'SENSOR_BLOCK',
            'MECHANICAL_JAM',
            'BLURRY_IMAGE',
            'QR_CODE_DAMAGED',
            'OBJECT_OUT_OF_BOUNDS',
            'TRAY_MALFUNCTION',
            'CALIBRATION_LOSS',
            'DIVERTER_ERROR'
        )
        THEN 'INVALID_ERROR_CODE'

    -- Status Validation
        WHEN e.status IS NULL 
            OR TRIM(e.status) = ''       
        THEN 'NULL_OR_EMPTY_STATUS'

        WHEN LOWER(TRIM(e.status)) = 'null'                   
        THEN 'STRING_NULL_IN_STATUS'

        WHEN UPPER(TRIM(e.status)) NOT IN (
            'OPEN',
            'RESOLVED'
        )
        THEN 'INVALID_STATUS'

    -- Timestamps Validation
        -- Reported At
        WHEN e.reported_at IS NULL 
            OR TRIM(e.reported_at) = ''       
        THEN 'NULL_OR_EMPTY_REPORTED_AT'

        WHEN LOWER(TRIM(e.reported_at)) = 'null'                   
        THEN 'STRING_NULL_IN_REPORTED_AT'

        WHEN TRY_CAST(e.reported_at AS TIMESTAMP) IS NULL
        THEN 'INVALID_REPORTED_AT_FORMAT'

        -- Resolved At
        WHEN LOWER(TRIM(e.resolved_at)) = 'null'                   
        THEN 'STRING_NULL_IN_RESOLVED_AT'

        WHEN e.resolved_at IS NOT NULL 
            AND TRIM(e.resolved_at) != ''
            AND LOWER(TRIM(e.resolved_at)) != 'null'
            AND TRY_CAST(TRIM(e.resolved_at) AS TIMESTAMP) IS NULL
        THEN 'INVALID_RESOLVED_AT_FORMAT'

        -- Business Logic Validation
        WHEN e.reported_at > e.resolved_at
        THEN 'INVALID_REPORTED_AT_TIME'

    -- Employee ID Validation
        WHEN e.employee_id IS NULL 
            OR TRIM(e.employee_id) = ''       
        THEN 'NULL_OR_EMPTY_EMPLOYEE_ID'

        WHEN LOWER(TRIM(e.employee_id)) = 'null'                   
        THEN 'STRING_NULL_IN_EMPLOYEE_ID'

        WHEN UPPER(TRIM(e.employee_id)) NOT LIKE 'EMP-%'
        THEN 'INVALID_EMPLOYEE_ID'

    -- Foreign Key Presence Validation
        WHEN d.device_id IS NULL                                 
        THEN 'FK_DEVICE_NOT_FOUND_IN_SILVER'

        WHEN p.parcel_id IS NULL                                 
        THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'

        ELSE NULL
    END AS _rejection_reason,

    _ingested_at,
    _source_file,
    current_timestamp() AS _validated_at

FROM parcel_sorting_hub.bronze.exceptions e

LEFT JOIN parcel_sorting_hub.silver.devices d 
    ON TRIM(e.device_id) = TRIM(d.device_id)

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON TRIM(e.parcel_id) = TRIM(p.parcel_id)

WHERE e._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.exceptions)

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.exceptions
SELECT 
    exception_id,        
    parcel_id,            
    device_id,             
    error_code,   
    status,
    reported_at,
    resolved_at,
    employee_id,               
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_exceptions
WHERE _rejection_reason IS NOT NULL

In [0]:
MERGE INTO parcel_sorting_hub.silver.exceptions AS target
USING (
    WITH deduped AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY TRIM(exception_id) ORDER BY _ingested_at DESC) AS row_num_id,
            ROW_NUMBER() OVER (PARTITION BY 
                TRIM(exception_id),
                TRIM(parcel_id),
                TRIM(device_id),
                TRIM(error_code),
                TRIM(status),
                TRIM(reported_at),
                TRIM(resolved_at),
                TRIM(employee_id)
            ORDER BY _ingested_at DESC) AS row_dup
        FROM validated_exceptions
        WHERE _rejection_reason IS NULL
    )
    
    SELECT
        TRIM(exception_id)                         AS exception_id,
        TRIM(parcel_id)                            AS parcel_id,
        TRIM(device_id)                            AS device_id,
        TRIM(error_code)                           AS error_code,
        TRIM(status)                               AS status,
        TRY_CAST(TRIM(reported_at) AS TIMESTAMP)   AS reported_at,
        TRY_CAST(TRIM(resolved_at) AS TIMESTAMP)   AS resolved_at,
        TRIM(employee_id)                          AS employee_id,
        current_timestamp()                        AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1
    AND 
        row_dup = 1

) AS source
ON target.exception_id = source.exception_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *     

### SORTING

In [0]:
CREATE OR REPLACE TEMP VIEW validated_sorting AS

SELECT 
    s.event_id,        
    s.parcel_id,            
    s.sorter_id,             
    s.chute_id,   
    s.entry_time,
    s.result,

    CASE
    -- Primary Key Validation
        WHEN s.event_id IS NULL 
            OR TRIM(s.event_id) = ''    
        THEN 'NULL_OR_EMPTY_PK'

        WHEN LOWER(TRIM(s.event_id)) = 'null'               
        THEN 'STRING_NULL_IN_PK'

        WHEN UPPER(TRIM(s.event_id)) NOT LIKE 'SRT-%'
        THEN 'INVALID_PK'

    -- Foreign Key Validation
        -- Parcel ID
        WHEN s.parcel_id IS NULL 
            OR TRIM(s.parcel_id) = '' 
        THEN 'NULL_OR_EMPTY_FK'

        WHEN LOWER(TRIM(s.parcel_id)) = 'null'                
        THEN 'STRING_NULL_IN_FK'

        WHEN LENGTH(TRIM(s.parcel_id)) != 24
        THEN 'INVALID_FK_PARCEL_ID'

        -- SORTER ID
        WHEN s.sorter_id IS NULL 
            OR TRIM(s.sorter_id) = ''  
        THEN 'NULL_OR_EMPTY_SORTER_ID'

        WHEN LOWER(TRIM(s.sorter_id)) = 'null'              
        THEN 'STRING_NULL_IN_SORTER_ID'

        WHEN UPPER(TRIM(s.sorter_id)) NOT LIKE 'DEV-%'
        THEN 'INVALID_FK_SORTER_ID'

        -- Chute ID
        WHEN s.chute_id IS NULL 
            OR TRIM(s.chute_id) = ''    
        THEN 'NULL_OR_EMPTY_CHUTE_ID'

        WHEN LOWER(TRIM(s.chute_id)) = 'null'               
        THEN 'STRING_NULL_IN_CHUTE_ID'

        WHEN UPPER(TRIM(s.chute_id)) NOT LIKE 'CHUTE-%'
        THEN 'INVALID_FK_CHUTE_ID'

    -- Timestampt Validation
        WHEN s.entry_time IS NULL 
            OR TRIM(s.entry_time) = ''    
        THEN 'NULL_OR_EMPTY_TIMESTAMP'

        WHEN LOWER(TRIM(s.entry_time)) = 'null'               
        THEN 'STRING_NULL_IN_TIMESTAMP'

        WHEN TRY_CAST(TRIM(s.entry_time) AS TIMESTAMP) IS NULL
        THEN 'INVALID_TIMESTAMP_FORMAT'
    
    -- Result Validation
        WHEN s.result IS NULL 
            OR TRIM(s.result) = ''    
        THEN 'NULL_OR_EMPTY_RESULT'

        WHEN LOWER(TRIM(s.result)) = 'null'
        THEN 'STRING_NULL_IN_RESULT'

        WHEN UPPER(TRIM(s.result)) NOT IN (
            'SUCCESS', 
            'REROUTED'
            )
        THEN 'INVALID_RESULT'

    -- Foreign Key Presence Validation
        WHEN p.parcel_id IS NULL                            
        THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'

        WHEN d.device_id IS NULL                            
        THEN 'FK_SORTER_NOT_FOUND_IN_SILVER'

        ELSE NULL
    END AS _rejection_reason,

    _ingested_at,
    _source_file,
    current_timestamp() AS _validated_at

FROM parcel_sorting_hub.bronze.sorting s

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON TRIM(s.parcel_id) = TRIM(p.parcel_id)

LEFT JOIN parcel_sorting_hub.silver.devices d  
    ON TRIM(s.sorter_id) = TRIM(d.device_id)

WHERE s._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.sorting)

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.sorting
SELECT 
    event_id,        
    parcel_id,            
    sorter_id,             
    chute_id,   
    entry_time,
    result,
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_sorting
WHERE _rejection_reason IS NOT NULL;

In [0]:
MERGE INTO parcel_sorting_hub.silver.sorting AS target
USING (
    WITH deduped AS (

            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY TRIM(event_id) ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    TRIM(event_id),
                    TRIM(parcel_id),
                    TRIM(sorter_id),
                    TRIM(chute_id), 
                    TRIM(entry_time),
                    TRIM(result)
                ORDER BY _ingested_at DESC) AS row_dup
            FROM validated_sorting
            WHERE _rejection_reason IS NULL
    )
    
    SELECT
        TRIM(event_id)                          AS event_id,
        TRIM(parcel_id)                         AS parcel_id,
        TRIM(sorter_id)                         AS sorter_id,
        TRIM(chute_id)                          AS chute_id,
        TRY_CAST(TRIM(entry_time) AS TIMESTAMP) AS entry_time,
        TRIM(result)                            AS result,
        current_timestamp()                     AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1
    AND 
        row_dup = 1

) AS source
ON target.event_id = source.event_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

### STATUS HISTORY

In [0]:
CREATE OR REPLACE TEMP VIEW validated_status_history AS 

SELECT 
    h.history_id,        
    h.parcel_id,            
    h.status_name,             
    h.change_timestamp,

    CASE
    -- Primary Key Validation
        WHEN h.history_id IS NULL 
            OR TRIM(h.history_id) = ''    
        THEN 'NULL_OR_EMPTY_PK'

        WHEN LOWER(TRIM(h.history_id)) = 'null'                 
        THEN 'STRING_NULL_IN_PK'

        WHEN UPPER(TRIM(h.history_id)) NOT LIKE 'ST-%'
        THEN 'INVALID_PK'

    -- Foreign Key Validation
        -- Parcel ID
        WHEN h.parcel_id IS NULL 
            OR TRIM(h.parcel_id) = '' 
        THEN 'NULL_OR_EMPTY_FK'

        WHEN LOWER(TRIM(h.parcel_id)) = 'null'                
        THEN 'STRING_NULL_IN_FK'

        WHEN LENGTH(TRIM(h.parcel_id)) != 24
        THEN 'INVALID_FK_PARCEL_ID'

    -- Status Name Validation
        WHEN h.status_name IS NULL 
            OR TRIM(h.status_name) = ''  
        THEN 'NULL_OR_EMPTY_STATUS_NAME'

        WHEN LOWER(TRIM(h.status_name)) = 'null'                
        THEN 'STRING_NULL_IN_STATUS_NAME'

        WHEN UPPER(TRIM(h.status_name)) NOT IN (
            'RECEIVED', 
            'SORTED', 
            'LOADED', 
            'EXCEPTION'
            )
        THEN 'INVALID_STATUS_NAME'

    -- Timestamp Validation
        WHEN h.change_timestamp IS NULL 
            OR TRIM(h.change_timestamp) = ''                    
        THEN 'NULL_OR_EMPTY_CHANGE_TIMESTAMP'

        WHEN LOWER(TRIM(h.change_timestamp)) = 'null'           
        THEN 'STRING_NULL_IN_CHANGE_TIMESTAMP'

        WHEN TRY_CAST(TRIM(h.change_timestamp) AS TIMESTAMP) IS NULL
        THEN 'INVALID_TIMESTAMP_FORMAT'

        WHEN p.parcel_id IS NULL                                
        THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'

        ELSE NULL
    END AS _rejection_reason,
    
    _ingested_at,
    _source_file,
    current_timestamp() AS _validated_at

FROM parcel_sorting_hub.bronze.status_history h

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON TRIM(h.parcel_id) = TRIM(p.parcel_id)

WHERE h._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.status_history)

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.status_history
SELECT 
    history_id,        
    parcel_id,            
    status_name,             
    change_timestamp,
    _source_file,                    
    _rejection_reason,
    current_timestamp() AS _rejected_at
FROM validated_status_history
WHERE _rejection_reason IS NOT NULL;

In [0]:
MERGE INTO parcel_sorting_hub.silver.status_history AS target
USING (
    WITH deduped AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY TRIM(history_id) ORDER BY _ingested_at DESC) AS row_num_id,
            ROW_NUMBER() OVER (PARTITION BY 
                TRIM(history_id),
                TRIM(parcel_id),
                TRIM(status_name),
                TRIM(change_timestamp)
            ORDER BY _ingested_at DESC) AS row_dup
        FROM validated_status_history
        WHERE _rejection_reason IS NULL
    )
    
    SELECT
        TRIM(history_id)                              AS history_id,
        TRIM(parcel_id)                               AS parcel_id,
        TRIM(status_name)                             AS status,
        TRY_CAST(TRIM(change_timestamp) AS TIMESTAMP) AS change_timestamp,
        current_timestamp()                           AS _updated_at
    FROM deduped
    WHERE 
        row_num_id = 1
    AND 
        row_dup = 1

) AS source
ON target.history_id = source.history_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *